# **LangChain Basics Tutorial**

## **1. Environment Setup**

**1-1. Install LangChain**

In [ ]:
!pip install langchain langchain-openai

from langchain_openai import OpenAIEmbeddings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.5 MB/s eta 0:00:00


**1-2. Set API Key**

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = "YOUR API KEY"

## **2. LLM Chains**

**2-1. Basic LLM Chain**

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# Run the chain
print(llm.invoke("What is the rotation period of the Earth?").content)


**2-2. Multi-Chain**

**Basic Sequential Chain**

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model("gpt-4o-mini")

# 2-step sequential chain: define → explain in detail
# Step 1: Define the term in one sentence
define_prompt = ChatPromptTemplate.from_template(
    "Define the following AI/ML term in one sentence: {term}"
)

# Step 2: Expand on the definition
explain_prompt = ChatPromptTemplate.from_template(
    "Expand on this definition with concrete examples and real-world use cases:\n{definition}"
)

# Chain 1: define
chain1 = define_prompt | llm | StrOutputParser()

# Chain 2: use chain1 output as input
chain2 = ({"definition": chain1} | explain_prompt | llm | StrOutputParser())

result = chain2.invoke({"term": "Retrieval Augmented Generation"})
print(result)


**Multi-step Sequential Chain**

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model("gpt-4o-mini")

# 3-step chain: extract keywords → outline → write article
analyze_prompt = ChatPromptTemplate.from_template(
    "Extract 3 key keywords from this topic: {topic}"
)
outline_prompt = ChatPromptTemplate.from_template(
    "Write an article outline based on these keywords: {keywords}\nOriginal topic: {topic}"
)
content_prompt = ChatPromptTemplate.from_template(
    "Write a concise ~200 word article based on this outline:\n{outline}"
)

chain = (
    {"topic": RunnablePassthrough()}
    | RunnablePassthrough.assign(keywords=analyze_prompt | llm | StrOutputParser())
    | RunnablePassthrough.assign(outline=outline_prompt | llm | StrOutputParser())
    | content_prompt | llm | StrOutputParser()
)

result = chain.invoke("The impact of LLMs on manufacturing automation")
print(result)


**Parallel Chain**

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

llm = init_chat_model("gpt-4o-mini")

# Analyze a topic from three angles simultaneously
positive_prompt = ChatPromptTemplate.from_template("List 3 positive aspects of {topic}.")
negative_prompt = ChatPromptTemplate.from_template("List 3 negative aspects or risks of {topic}.")
neutral_prompt  = ChatPromptTemplate.from_template("Describe the current state of {topic} objectively.")

# All three chains run in parallel
parallel_chain = RunnableParallel(
    positive=positive_prompt | llm | StrOutputParser(),
    negative=negative_prompt | llm | StrOutputParser(),
    neutral =neutral_prompt  | llm | StrOutputParser(),
)

results = parallel_chain.invoke({"topic": "AI's impact on the manufacturing industry"})

print("=== Positive Aspects ===")
print(results["positive"])
print("\n=== Negative Aspects / Risks ===")
print(results["negative"])
print("\n=== Current State ===")
print(results["neutral"])


## **3. Prompts**

**Clarity and Specificity**

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model("gpt-4o-mini")

# Vague prompt
vague_prompt = ChatPromptTemplate.from_template("Tell me about {topic}.")

# Specific, structured prompt
specific_prompt = ChatPromptTemplate.from_template(
"""
Explain {topic} using the following format:

1. Definition (1-2 sentences)
2. Three key characteristics
3. Two real-world applications

Use plain language that a beginner can understand.
""")


In [ ]:
# Response to the vague prompt
chain = vague_prompt | llm
result = chain.invoke({"topic": "artificial intelligence"})
print(result.content)


In [ ]:
# Response to the specific prompt
chain = specific_prompt | llm
result = chain.invoke({"topic": "artificial intelligence"})
print(result.content)


**Providing Context**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Prompt without context
without_context = "Recommend a Python web framework."

# Prompt with context
with_context = ChatPromptTemplate.from_messages([
    ("system", "You are a senior backend developer."),
    ("human", """Recommend a Python web framework for the following project:

Project details:
- Team size: 3 (2 junior, 1 senior)
- Expected traffic: 100,000 requests/day
- Key features: REST API, real-time notifications
- Deadline: 3 months

Explain the pros and cons of each option and why it suits this project.""")
])


In [ ]:
# Response without context
result = chain.invoke(without_context)
print(result.content)


In [ ]:
# Response with context
result = chain.invoke(with_context)
print(result.content)


**Structured Prompts**

In [ ]:
# Role assignment via system message
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model("gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a senior data scientist with 10 years of experience.

Expertise:
- Machine learning model development
- Data analysis and visualization
- Python, SQL, TensorFlow

Response style:
- Data-driven explanations
- Include code examples where relevant
- Practical, real-world advice"""),
    ("human", "{question}")
])

question = "What is the best approach for data visualization in Python?"
chain = prompt | llm
result = chain.invoke({"question": question})
print(result.content)


**Specifying Output Format**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Markdown table format
table_prompt = ChatPromptTemplate.from_messages([
    ("system", "Format your response as a markdown table."),
    ("human", """Compare the following programming languages: {languages}

Use this format:
| Language | Main Use | Pros | Cons | Learning Curve |""")
])

# JSON format
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Respond with valid JSON only. No other text."),
    ("human", """Extract information from this text: {text}

Format:
{{
    "name": "...",
    "email": "...",
    "phone": "..."
}}
""")
])

# Numbered list format
list_prompt = ChatPromptTemplate.from_messages([
    ("system", "Format your response as a numbered list. Keep each item concise."),
    ("human", "List 5 advantages of {topic}.")
])


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model("gpt-4o-mini")

table_prompt = ChatPromptTemplate.from_messages([
    ("system", "Format your response as a markdown table."),
    ("human", """Compare the following programming languages: {languages}

Use this format:
| Language | Main Use | Pros | Cons | Learning Curve |""")
])

languages = "Python, Java, C++"
chain = table_prompt | llm
result = chain.invoke({"languages": languages})
print(result.content)


**Providing Examples (Few-shot)**

In [ ]:
from langchain_core.prompts import PromptTemplate

# Define a prompt template with two variables
template_text = "Hello, my name is {name} and I am {age} years old."
prompt_template = PromptTemplate.from_template(template_text)

# Fill in the template
filled_prompt = prompt_template.format(name="Alice", age=30)
print(filled_prompt)


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

result = model.invoke("What is the rotation period of the Earth?")
print(result.content)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

examples = [
    {"input": "What gas makes up the largest proportion of Earth's atmosphere?",
     "output": "Nitrogen."},
    {"input": "What are the main inputs required for photosynthesis?",
     "output": "Light, carbon dioxide, and water."},
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an educator well-versed in science and mathematics."),
    few_shot_prompt,
    ("human", "{input}"),
])

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
chain = final_prompt | model

result = chain.invoke({"input": "What is the largest planet in the solar system?"})
print(result.content)


In [ ]:
from langchain_chroma import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

examples = [
    {"input": "What gas makes up the largest proportion of Earth's atmosphere?",
     "output": "Nitrogen."},
    {"input": "What are the main inputs required for photosynthesis?",
     "output": "Light, carbon dioxide, and water."},
    {"input": "Explain the Pythagorean theorem.",
     "output": "In a right triangle, the square of the hypotenuse equals the sum of the squares of the other two sides."},
    {"input": "Briefly explain the basic structure of DNA.",
     "output": "DNA is a nucleic acid with a double helix structure."},
    {"input": "What is the definition of pi (π)?",
     "output": "The ratio of a circle's circumference to its diameter."},
    {"input": "How many times larger is Jupiter's volume than Earth's?",
     "output": "Approximately 1,321 times."},
]

to_vectorize = [" ".join(ex.values()) for ex in examples]
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_texts(to_vectorize, embeddings, metadatas=examples)

example_selector = SemanticSimilarityExampleSelector(vectorstore=vectorstore, k=2)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=ChatPromptTemplate.from_messages(
        [("human", "{input}"), ("ai", "{output}")]
    ),
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an educator well-versed in science and mathematics."),
    few_shot_prompt,
    ("human", "{input}"),
])

chain = final_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

question = "What is the largest planet in the solar system?"
selected_examples = example_selector.select_examples({"input": question})

print("=== Selected Examples ===")
for i, ex in enumerate(selected_examples, 1):
    print(f"[Example {i}]")
    print("Input :", ex["input"])
    print("Output:", ex["output"])
    print()

result = chain.invoke(question)
print(result.content)


## **4. Memory**

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver

model = init_chat_model("openai:gpt-4o")

# Without memory — the model forgets between calls
without_memory1 = model.invoke("Hello, my name is Alice.")
print(without_memory1.content)

without_memory2 = model.invoke("What is my name?")
print(without_memory2.content)


In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
model = init_chat_model("openai:gpt-4o")

# With memory — the agent remembers the thread
agent = create_agent(
    model,
    tools=[],
    checkpointer=checkpointer
)

config = {"configurable": {"thread_id": "conversation-1"}}

result1 = agent.invoke(
    {"messages": [("user", "Hello, my name is Alice.")]},
    config=config
)
print(result1["messages"][-1].content)

result2 = agent.invoke(
    {"messages": [("user", "What is my name?")]},
    config=config
)
print(result2["messages"][-1].content)


## **5. Building a Chatbot**

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver

SYSTEM_PROMPT = """You are 'SciBot', an AI research assistant specializing in science and engineering.
Your goal is to help researchers obtain accurate, reliable answers grounded in scientific reasoning.

[Role]
- Assist with research questions spanning physics, chemistry, biology, data science, and AI/ML.
- Support hypothesis formulation, experimental design, paper interpretation, result analysis, and methodology selection.
- Act as a critical colleague who reviews the reasoning process alongside the researcher.

[Answer Principles]
1. Evidence first: back every claim with its underlying principle, mechanism, or data.
2. Accuracy over completeness: distinguish what is known from what is uncertain; acknowledge gaps openly.
3. No speculation as fact: label unverified content as an estimate or hypothesis.
4. Quantitative thinking: use equations, units, and order-of-magnitude reasoning where appropriate.
5. State limitations: mention assumptions, constraints, and possible counterexamples.

[Reasoning Style]
- Break complex problems into logical sub-steps.
- Make the flow assumption → reasoning → conclusion explicit.
- When multiple interpretations exist, compare leading hypotheses and evaluate evidence for each.

[Output Format]
- Respond in English; define technical terms on first use.
- Use clear notation for equations; annotate code snippets briefly.
- Lead with the key point, then supporting evidence and detail.
- Flag uncertain claims at the end with "Needs verification" or "Further confirmation recommended".

[Tone]
- Maintain context across the conversation and respond consistently.
- Correct mistaken assumptions respectfully, with explanation.
- Be critical but constructive — the tone of a good research collaborator.
"""

checkpointer = MemorySaver()
model = init_chat_model("openai:gpt-4o")

agent = create_agent(
    model,
    tools=[],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)


In [ ]:
config = {"configurable": {"thread_id": "chat-session"}}

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("exit", "quit"):
        break
    result = agent.invoke(
        {"messages": [("user", user_input)]},
        config=config,
    )
    print("SciBot:", result["messages"][-1].content)


In [ ]:
# ── 저장된 메모리 확인 ──
state = agent.get_state(config)
for m in state.values["messages"]:
    m.pretty_print()

================================ Human Message =================================

안녕 내 이름은 철수고 인공지능을 전공했어, 앞으로 내가 연구를 수행함에 있어 도움을 주면 좋겠어
================================== Ai Message ==================================

안녕하세요, 철수님! 만나서 반갑습니다. 인공지능 분야에서 연구를 준비하고 계시다니 흥미롭습니다. 어떤 주제나 문제를 탐구하고 계신지, 혹은 도움을 필요로 하는 특정 부분이 있는지 말씀해주시면 도움이 될 수 있도록 최선을 다하겠습니다. 함께 과학적 사고를 통해 목표를 이루어가요!
================================ Human Message =================================

내가 전공한 분야가 뭐라고?
================================== Ai Message ==================================

철수님은 인공지능을 전공하셨다고 말씀하셨어요. 인공지능 분야의 연구에 도움을 드리도록 하겠습니다. 특정한 질문이나 주제가 있으면 편하게 말씀해 주세요.
================================ Human Message =================================

LLM에 대해서 설명해줄래?
================================== Ai Message ==================================

물론입니다! LLM은 "Large Language Model"의 약자로, 대규모 언어 모델을 의미합니다. 이는 인간의 언어를 이해하고 생성하기 위해 설계된 인공지능 모델입니다. 다음은 LLM의 구성과 기능에 대한 간단한 설명입니다:

1. **기본 원리**:
   - LLM은 주로 딥러닝(deep learning) 기술